In [1]:
# %% 0 - percorsi (STESSI NOMI di 09C_06/09C_07, cosi' le celle di 09C_07 girano senza modifiche)
import sys, json, shutil, subprocess
from pathlib import Path
import pandas as pd
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark
from src.evaluation import distance_presets as dp
from src.evaluation import colosseum_controls as cc
from src.evaluation.json_ingest import ingest_chunks_json, load_tags_into_sidecar
pd.set_option("display.width", 250, "display.max_columns", 30)
 
COLLECTION = "colosseum_bench"                      # = COLLECTION_NAME in src/config.py (invariato)
Q_OLD = project_root / "data/queries/colosseum"     # file di 09C_06 (sorgente)
QDIR = project_root / "data/queries/colosseum_v2"   # file nuovi
sidecar_path = project_root / "data/processed/colosseum/colosseum_v2.json"
base_sidecar = sidecar_path.with_name("colosseum_v2_base.json")
chunks_path, queries_path = QDIR / "chunks_colosseum_final.json", QDIR / "eval_queries.json"
pred_path, norm_path, gold_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json", QDIR / "gold_chunks.json"
OUT_DIR = project_root / "data/benchmark/colosseum_v2"
TOP_J, TOP_K, TOP_N = 5, 15, 7                      # regime di partenza: lo sweep di 09C_07 (cella 4) dice se cambiarlo

In [ ]:
# %% 1 - genera i file v2 (monumento sui chunk e come conv_tag di OGNI query). Stampa la tabella: RIVEDI i conv_tag assegnati.
# Regole: monumento = quello del PRIMO chunk gold; conv_tag = [monumento, aspetto] per le ambigue (tranne AMB_04/05, vedi MULTI_CONV_EXCLUDE in add_monument_tags.py).
src = project_root / "src/evaluation"
r = subprocess.run([sys.executable, str(src / "add_monument_tags.py"), "--chunks", str(Q_OLD / "chunks_colosseum_final.json"),
                    "--out", str(QDIR), "--make_queries", str(src / "make_queries.py")], capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])

In [2]:
# %% 2 - ingestione (collezione + tag nel sidecar v2) e handler/bench. UNA volta; poi gli stessi file servono a tutte le celle.
sidecar_path.parent.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)
ingest_chunks_json(chunks_path, COLLECTION, sidecar_path).release_path()      # il sidecar v2 e' nuovo: nessuna distanza ne' predq ereditata
def make_bench(sp, h=None):
    h = h or PipelineHandler(sidecar_path=sp, queries_path=queries_path)
    return RagasBenchmark(h, predetermined_path=pred_path, normal_path=norm_path, out_dir=OUT_DIR,
                          top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
bench = make_bench(sidecar_path); handler = bench.h; sm = handler.sidecar_manager
allq = json.load(open(queries_path, encoding="utf-8")); gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
TAGS = sorted({t for c in chunks.values() for t in c["tags"]})
assert all(cc.conv_of(q) and cc.conv_of(q)[0] in cc.MONUMENT_TAGS for q in allq), "ogni query deve avere un monumento come primo conv_tag"
assert all(set(cc.conv_of(q)[1:]) <= set(TAGS) - set(cc.MONUMENT_TAGS) and set(q["tags"]) <= set(TAGS) - set(cc.MONUMENT_TAGS) for q in allq)
print("query con piu' conv tag:", sum(len(cc.conv_of(q)) > 1 for q in allq), "di", len(allq))
print({k: v["count"] for k, v in sm.get_global_tags().items()})

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


!>> 120 chunk in 'colosseum_bench' (19.5s); tag nel sidecar: ['Colosseum', 'History', 'Architecture', 'Culture', 'Spectacles', 'Conservation', 'Archaeology', 'DomusAurea', 'ColossusOfNero', 'LudusMagnus', 'ArchOfConstantine']
query con piu' conv tag: 12 di 42
{'Colosseum': 68, 'History': 48, 'Architecture': 48, 'Culture': 19, 'Spectacles': 28, 'Conservation': 14, 'Archaeology': 29, 'DomusAurea': 20, 'ColossusOfNero': 4, 'LudusMagnus': 11, 'ArchOfConstantine': 17}


In [3]:
# %% 3 - predq + similarita' + backup base (tag + predq, nessuna distanza)
print(len(bench.register_predetermined("pred")), "predq registrate")
display(bench.check_similarity(warn_above=0.80).head(6))
shutil.copy(sidecar_path, base_sidecar)

12 predq registrate


!> 1 query normali troppo simili a una predeterminata: ['MUL_01']


,normal_id,max_sim,closest_pred
20,MUL_01,0.826329,MUL_09
14,SPE_08,0.783214,AMB_05
25,MUL_07,0.777352,AMB_05
9,AMB_14,0.775193,AMB_05
19,SPE_13,0.761778,MUL_12
21,MUL_03,0.752021,SPE_02


PosixPath('/home/jovyan/tesi_graphrag/data/processed/colosseum/colosseum_v2_base.json')

In [4]:
# %% 4 - versione ESPERTA: stesso piano di distanze di 09C_06 (dipende da gold e chunk, non dai tag) + retrieval + risposte
plan_expert_path = Q_OLD / "distance_plan_colosseum_expert.json"                # il file che mi hai mandato
plan_expert = dp.load_plan(plan_expert_path)
assert {k.replace("predq_", "") for k in plan_expert["predq"]} == {q["id"] for q in json.load(open(pred_path))}, "predq del piano != predeterminate v2"
print(dp.apply_plan(sm, plan_expert))
genA = bench.generate("pilot_expert_v2", with_answers=False)       # retrieval soltanto (~1 min): prima di spendere LLM
display(bench.retrieval_summary("pilot_expert_v2"))
META = ["run_id", "query_id", "category", "subtype", "is_predetermined", "config"]; GOLD = ["gold_hit", "gold_recall", "gold_precision", "gold_mrr"]
mA = genA[META + GOLD]; normal = ~mA.is_predetermined
for lab, sub in (("NORMALI", normal), ("AMBIGUE normali", normal & (mA.category == "ambiguous"))):
    r = bench.paired_vs_baseline(mA, subset=sub); print("---", lab); display(r[r.metric.isin(["gold_recall", "gold_mrr"])])
# gen = bench.generate("post_edit_expert_v2")                       # con risposte: SOLO dopo aver scelto (top_k, top_j) con 09C_07 cella 4

<<| Batch adiacenze query pre-determinate salvato: 99 elementi processati |>>
<<| Batch adiacenze standard salvato: 499 elementi processati |>>
{'predq_edits': 99, 'chunk_pairs': 499}


gold_hit  gold_recall  gold_precision  gold_mrr
config                category                                                     
C0_baseline           ambiguous        0.714        0.345           0.200     0.595
                      multi_domain     0.786        0.448           0.343     0.577
                      specialistic     0.786        0.726           0.257     0.643
C1_tags               ambiguous        1.000        0.798           0.471     0.929
                      multi_domain     0.857        0.613           0.471     0.685
                      specialistic     1.000        0.833           0.300     0.756
C2_dist               ambiguous        1.000        0.798           0.471     0.929
                      multi_domain     0.857        0.595           0.457     0.714
                      specialistic     0.929        0.798           0.286     0.738
C3_dist_tags          ambiguous        1.000        0.821           0.486     0.929
                      multi_domain     0.857        0.595           0.457     0.685
                      specialistic     1.000        0.869           0.314     0.756
C4_dist_tags_joinconv ambiguous        1.000        0.869           0.514     0.929
                      multi_domain     0.929        0.756           0.571     0.708
                      specialistic     1.000        0.869           0.314     0.756
C5_dist_tags_joinall  ambiguous        1.000        0.762           0.443     0.907
                      multi_domain     0.929        0.732           0.557     0.702
                      specialistic     1.000        0.893           0.329     0.756

--- NORMALI


,metric,config,n,n_nan,mean_diff,ci95_low,ci95_high,win,tie,loss
5,gold_recall,C1_tags,30,0,0.250,0.139,0.369,15,14,1
6,gold_recall,C2_dist,30,0,0.233,0.128,0.350,13,17,0
7,gold_recall,C3_dist_tags,30,0,0.278,0.158,0.408,15,14,1
8,gold_recall,C4_dist_tags_joinconv,30,0,0.278,0.175,0.392,17,13,0
9,gold_recall,C5_dist_tags_joinall,30,0,0.200,0.119,0.294,15,15,0
15,gold_mrr,C1_tags,30,0,0.136,0.039,0.242,10,17,3
16,gold_mrr,C2_dist,30,0,0.142,0.058,0.239,9,20,1
17,gold_mrr,C3_dist_tags,30,0,0.136,0.039,0.242,10,17,3
18,gold_mrr,C4_dist_tags_joinconv,30,0,0.147,0.050,0.253,11,17,2
19,gold_mrr,C5_dist_tags_joinall,30,0,0.134,0.037,0.238,11,17,2


--- AMBIGUE normali


,metric,config,n,n_nan,mean_diff,ci95_low,ci95_high,win,tie,loss
5,gold_recall,C1_tags,10,0,0.533,0.333,0.733,9,1,0
6,gold_recall,C2_dist,10,0,0.533,0.333,0.733,9,1,0
7,gold_recall,C3_dist_tags,10,0,0.567,0.367,0.733,9,1,0
8,gold_recall,C4_dist_tags_joinconv,10,0,0.533,0.333,0.733,9,1,0
9,gold_recall,C5_dist_tags_joinall,10,0,0.333,0.167,0.533,7,3,0
15,gold_mrr,C1_tags,10,0,0.300,0.100,0.500,5,5,0
16,gold_mrr,C2_dist,10,0,0.300,0.100,0.500,5,5,0
17,gold_mrr,C3_dist_tags,10,0,0.300,0.100,0.500,5,5,0
18,gold_mrr,C4_dist_tags_joinconv,10,0,0.300,0.100,0.500,5,5,0
19,gold_mrr,C5_dist_tags_joinall,10,0,0.270,0.090,0.490,5,5,0


In [5]:
# %% 5 - versione UTENTE MEDIO a due assi (aspetto: 30% mancanti/15% sbagliati/solo primo; monumento: 10% mancante/5% sbagliato)
KEEP, SEED = 0.6, 7
plan_user = dp.degrade_plan(plan_expert, keep=KEEP, seed=SEED)
topical = [t for t in TAGS if t not in cc.MONUMENT_TAGS]
kt, km = cc.VARIANTS["combinato_user"]
chunks_user, log = cc.degrade_two_axes(list(chunks.values()), topical, topical=kt, monument=km, seed=SEED)
print(log)
user_sidecar = sidecar_path.with_name("colosseum_v2_user.json"); shutil.copy(base_sidecar, user_sidecar)
load_tags_into_sidecar(chunks_user, user_sidecar).release_path()
bench_u = make_bench(user_sidecar); print(dp.apply_plan(bench_u.h.sidecar_manager, plan_user))
# gen_u = bench_u.generate("post_edit_user_v2")                      # con risposte, come sopra

{'secondary_dropped': 46, 'wrong': 12, 'missing': 39, 'monument_wrong': 5, 'monument_missing': 16}


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


<<| Batch adiacenze query pre-determinate salvato: 59 elementi processati |>>
<<| Batch adiacenze standard salvato: 299 elementi processati |>>
{'predq_edits': 59, 'chunk_pairs': 299}


## Controllo celle
Commenti per funzionamento

In [6]:
# %% 0 - setup
import importlib, json, shutil
import pandas as pd
from src.evaluation import colosseum_controls as cc
importlib.reload(cc)
pd.set_option("display.width", 250, "display.max_columns", 30)
gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
TAGS = sorted({t for c in chunks.values() for t in c["tags"]})
exp_gen = bench.load_generation("pilot_expert_v2")          # riferimento: retrieval esperto v2 con tag giusti (cella 4 di 09C_08), stesso regime (top_k=15, top_j=5)
print(len(allq), "query;", len(chunks), "chunk;", TAGS)
assert set(cc.MONUMENT_TAGS) <= set(TAGS), "dati v1: esegui prima 09C_08 (celle 0-4) in questo kernel"
assert "colosseum_v2" in str(chunks_path), f"chunks_path punta ai file v1: {chunks_path}"

42 query; 120 chunk; ['ArchOfConstantine', 'Archaeology', 'Architecture', 'Colosseum', 'ColossusOfNero', 'Conservation', 'Culture', 'DomusAurea', 'History', 'LudusMagnus', 'Spectacles']


In [7]:
# %% 1 - CONTROLLO NEGATIVO su entrambi gli assi (ca. 2 min). Atteso: il recall CALA rispetto al run con tag giusti.
#   monument: conv_tag = ALTRO monumento (aspetto giusto) | aspect: conv_tag giusto, aspetto (tag manuale) sbagliato
for mode in ("monument", "aspect"):
    wq = cc.wrong_tag_queries(bench.queries, gold, chunks, TAGS, seed=0, mode=mode)
    wrong = cc.run(bench, f"ctrl_wrong_{mode}", queries=wq)
    print(f"=== {mode}: quota dei top-J che porta il tag sbagliato usato"); display(cc.wrong_tag_leak(wrong, wq, chunks, mode=mode).to_frame().T)
    t = cc.paired_between_runs(exp_gen, wrong, subsets=("normali", "ambigue normali"))    # B - A = tag sbagliato - tag giusto
    display(t[t.metric == "gold_recall"].drop(columns="metric"))

=== monument: quota dei top-J che porta il tag sbagliato usato


config,C0_baseline,C1_tags,C2_dist,C3_dist_tags,C4_dist_tags_joinconv,C5_dist_tags_joinall
quota_con_tag_usato,0.04,0.53,0.52,0.53,0.53,0.15


,subset,config,n,mean_A,mean_B,diff_B_minus_A,ci95_low,ci95_high,win,tie,loss
0,normali,C0_baseline,30,0.522,0.522,0.000,0.000,0.000,0,30,0
2,normali,C1_tags,30,0.772,0.281,-0.492,-0.633,-0.347,0,10,20
4,normali,C2_dist,30,0.756,0.233,-0.522,-0.664,-0.381,0,9,21
6,normali,C3_dist_tags,30,0.800,0.281,-0.519,-0.667,-0.369,0,10,20
8,normali,C4_dist_tags_joinconv,30,0.800,0.303,-0.497,-0.636,-0.353,0,9,21
10,normali,C5_dist_tags_joinall,30,0.722,0.561,-0.161,-0.233,-0.094,0,17,13
12,ambigue normali,C0_baseline,10,0.333,0.333,0.000,0.000,0.000,0,10,0
14,ambigue normali,C1_tags,10,0.867,0.433,-0.433,-0.667,-0.200,0,4,6
16,ambigue normali,C2_dist,10,0.867,0.267,-0.600,-0.800,-0.400,0,1,9
18,ambigue normali,C3_dist_tags,10,0.900,0.433,-0.467,-0.733,-0.200,0,4,6


=== aspect: quota dei top-J che porta il tag sbagliato usato


config,C0_baseline,C1_tags,C2_dist,C3_dist_tags,C4_dist_tags_joinconv,C5_dist_tags_joinall
quota_con_tag_usato,0.16,0.35,0.29,0.34,0.4,0.31


,subset,config,n,mean_A,mean_B,diff_B_minus_A,ci95_low,ci95_high,win,tie,loss
0,normali,C0_baseline,30,0.522,0.522,0.000,0.000,0.000,0,30,0
2,normali,C1_tags,30,0.772,0.503,-0.269,-0.400,-0.147,1,15,14
4,normali,C2_dist,30,0.756,0.511,-0.244,-0.389,-0.111,0,21,9
6,normali,C3_dist_tags,30,0.800,0.492,-0.308,-0.442,-0.178,1,13,16
8,normali,C4_dist_tags_joinconv,30,0.800,0.514,-0.286,-0.403,-0.175,0,14,16
10,normali,C5_dist_tags_joinall,30,0.722,0.558,-0.164,-0.242,-0.094,0,17,13
12,ambigue normali,C0_baseline,10,0.333,0.333,0.000,0.000,0.000,0,10,0
14,ambigue normali,C1_tags,10,0.867,0.200,-0.667,-0.833,-0.467,0,1,9
16,ambigue normali,C2_dist,10,0.867,0.133,-0.733,-0.933,-0.500,0,1,9
18,ambigue normali,C3_dist_tags,10,0.900,0.167,-0.733,-0.900,-0.567,0,0,10


In [8]:
# %% 2 - DUE LETTURE: stessa query, tag A / tag B (bozza in cc.TWO_READINGS: RIVEDILA leggendo i chunk prima di fidarti)
pairs = cc.TWO_READINGS
cc.validate_two_readings(pairs, gold, chunks)
for qid, p in pairs.items():
    q = next(q for q in allq if q["id"] == qid)
    print(f"{qid}: {q['query']}\n   A={q['expected_tag']} {gold[qid]}\n   B={p['tag_b']} {p['gold_b']}")
tq = cc.two_readings_queries(bench.queries, pairs)
tr = cc.run(bench, "ctrl_tworeadings", queries=tq)
rep, det = cc.two_readings_report(tr, pairs, gold)
display(rep)         # specificita' = R_own - R_other: C0 = 0 per costruzione; con i tag deve diventare > 0
display(det[det.config.isin(["C0_baseline", "C1_tags", "C4_dist_tags_joinconv"])].pivot_table(index=["query", "lettura"], columns="config", values=["R_own", "R_other"]).round(2))

AMB_14: What kind of events take place at the Colosseum?
   A=['Colosseum', 'Culture'] ['wiki_colosseum_15', 'wiki_colosseum_30', 'wiki_colosseum_02']
   B=Spectacles ['wiki_colosseum_27', 'wiki_colosseum_10', 'wiki_colosseum_29']
AMB_12: Were there ships in the Colosseum?
   A=['Colosseum', 'Spectacles'] ['wiki_colosseum_28', 'wiki_naumachia_09', 'wiki_naumachia_08']
   B=Archaeology ['web_underground_04', 'web_underground_03', 'wiki_colosseum_24']
AMB_07: What happened to the frescoes of the Golden House?
   A=['DomusAurea', 'Culture'] ['wiki_domus_aurea_05', 'wiki_domus_aurea_04', 'wiki_domus_aurea_14']
   B=Conservation ['wiki_domus_aurea_06', 'wiki_domus_aurea_20']
AMB_09: Tell me about the Ludus Magnus.
   A=['LudusMagnus', 'Archaeology'] ['wiki_ludus_magnus_11', 'wiki_ludus_magnus_06', 'wiki_ludus_magnus_09']
   B=History ['wiki_ludus_magnus_04', 'wiki_ludus_magnus_05', 'wiki_ludus_magnus_01']
AMB_08: Where did the gladiators live and train?
   A=['LudusMagnus', 'Architecture'] 

,R_own,R_other,specificita
config,,,
C0_baseline,0.233,0.233,0.000
C1_tags,0.767,0.250,0.517
C2_dist,0.833,0.167,0.667
C3_dist_tags,0.833,0.167,0.667
C4_dist_tags_joinconv,0.767,0.200,0.567
C5_dist_tags_joinall,0.500,0.333,0.167


R_other                                     R_own                              
config         C0_baseline C1_tags C4_dist_tags_joinconv C0_baseline C1_tags C4_dist_tags_joinconv
query  lettura                                                                                    
AMB_07 A              0.00    0.50                  0.00        0.33    1.00                  1.00
       B              0.33    0.33                  0.33        0.00    0.00                  0.00
AMB_08 A              0.67    0.33                  0.33        0.00    1.00                  1.00
       B              0.00    0.00                  0.00        0.67    1.00                  1.00
AMB_09 A              0.33    0.33                  0.33        0.33    1.00                  1.00
       B              0.33    0.33                  0.33        0.33    0.67                  0.67
AMB_12 A              0.00    0.33                  0.33        0.00    1.00                  1.00
       B              0.00    0.33                  0.33        0.00    0.67                  0.67
AMB_14 A              0.00    0.00                  0.00        0.67    0.67                  0.67
       B              0.67    0.00                  0.00        0.00    0.67                  0.67

In [9]:
# %% 3 - DEGRADAZIONE TAG SEPARATA (6 varianti x 3 seed; sidecar BASE = tag + predq, nessuna distanza). ~15 min.
from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark
def make_bench(sp):
    h = PipelineHandler(sidecar_path=sp, queries_path=queries_path)
    return RagasBenchmark(h, predetermined_path=pred_path, normal_path=norm_path, out_dir=OUT_DIR,
                          top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
deg = cc.degradation_study(chunks, TAGS, base_sidecar, work_dir=sidecar_path.parent / "deg", make_bench=make_bench, seeds=(0, 1, 2))
deg.to_csv(OUT_DIR / "ctrl_degradation_retrieval.csv", index=False)
display(cc.summarize_vs_c0(deg))     # differenza con C0 (media|std sui seed): quale errore dei tag fa piu' danno?

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


[tag_esperti s0] {} | senza monumento: 0/120


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


[aspetti_mancanti30 s0] {'missing': 33} | senza monumento: 0/120


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


...

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


[combinato_user s1] {'missing': 37, 'secondary_dropped': 46, 'wrong': 8, 'monument_missing': 15, 'monument_wrong': 3} | senza monumento: 15/120


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


[combinato_user s2] {'secondary_dropped': 44, 'missing': 37, 'wrong': 12, 'monument_wrong': 11, 'monument_missing': 12} | senza monumento: 12/120


,variant,subset,metric,C1_tags|mean,C1_tags|std,C4_dist_tags_joinconv|mean,C4_dist_tags_joinconv|std,C5_dist_tags_joinall|mean,C5_dist_tags_joinall|std
0,aspetti_mancanti30,normali,gold_recall,0.149,0.018,0.156,0.022,0.126,0.011
1,aspetti_sbagliati15,normali,gold_recall,0.213,0.006,0.232,0.014,0.160,0.010
2,aspetti_sbagliati30,normali,gold_recall,0.133,0.031,0.152,0.038,0.093,0.052
3,combinato_user,normali,gold_recall,0.019,0.045,0.045,0.052,0.090,0.004
4,monumento_mancante20,normali,gold_recall,0.122,0.029,0.146,0.032,0.165,0.020
5,monumento_sbagliato10,normali,gold_recall,0.204,0.021,0.210,0.015,0.161,0.017
6,solo_primo_aspetto,normali,gold_recall,0.219,0.000,0.236,0.000,0.211,0.000
7,tag_esperti,normali,gold_recall,0.250,NaN,0.278,NaN,0.200,NaN
8,aspetti_mancanti30,normali,gold_mrr,0.090,0.052,0.114,0.056,0.095,0.052
9,aspetti_sbagliati15,normali,gold_mrr,0.144,0.037,0.155,0.037,0.138,0.040


In [10]:
# %% 4 - DISTANZE: sidecar CON distanze (esperto) contro SENZA distanze (stessi tag), per regime (top_k, top_j). ~15 min.
# Regimi: (15,5) (10,5) (7,5) = pochi posti liberi; (5,8) (5,10) = posti liberi sempre >= 3 (i 5 vettoriali fanno da seme, il grafo riempie il resto).
# C0 e' sempre il vettoriale puro a top_j chunk (run_regime), cosi' il confronto e' alla pari.
nodist_sidecar = sidecar_path.with_name("colosseum_benchmark_nodist.json")
shutil.copy(base_sidecar, nodist_sidecar)                 # base = tag esperti + predq, nessuna adiacenza
bench_nd = make_bench(nodist_sidecar)
tab, gens = cc.distance_ablation(bench, bench_nd)
tab.to_csv(OUT_DIR / "ctrl_distance_ablation.csv", index=False)
view = tab[(tab.metric == "gold_recall")].drop(columns="metric")
print("EFFETTO DELLE DISTANZE (con - senza), gold_recall; quota_query_cambiate = normali in cui cambiano i top-J")
display(view[view.config != "C0_baseline"].sort_values(["subset", "top_k", "top_j", "config"]))
print("config contro C0 (diff_vs_C0), SENZA distanze (side=0) e CON distanze (side=1):")
display(cc.vs_c0_by_topk(gens, side=0)); display(cc.vs_c0_by_topk(gens, side=1))
print("quota di query con contesto CORTO (< top_j chunk: categoria 1 vuota, distanze non partite), regime (5,8), con distanze:")
display(cc.short_context_quota(gens[(5, 8)][1], 8))

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


EFFETTO DELLE DISTANZE (con - senza), gold_recall; quota_query_cambiate = normali in cui cambiano i top-J


,subset,config,n,mean_A,mean_B,diff_B_minus_A,ci95_low,ci95_high,win,tie,loss,top_k,top_j,quota_query_cambiate
72,ambigue normali,C1_tags,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,8,0.0
74,ambigue normali,C3_dist_tags,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,8,0.0
76,ambigue normali,C4_dist_tags_joinconv,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,8,0.0
78,ambigue normali,C5_dist_tags_joinall,10,0.667,0.667,0.0,0.0,0.0,0,10,0,5,8,0.0
92,ambigue normali,C1_tags,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,10,0.0
94,ambigue normali,C3_dist_tags,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,10,0.0
96,ambigue normali,C4_dist_tags_joinconv,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,10,0.0
98,ambigue normali,C5_dist_tags_joinall,10,0.700,0.700,0.0,0.0,0.0,0,10,0,5,10,0.0
52,ambigue normali,C1_tags,10,0.800,0.800,0.0,0.0,0.0,0,10,0,7,5,0.0
54,ambigue normali,C3_dist_tags,10,0.833,0.833,0.0,0.0,0.0,0,10,0,7,5,0.0


config contro C0 (diff_vs_C0), SENZA distanze (side=0) e CON distanze (side=1):


,top_k,top_j,subset,config,gold_recall,diff_vs_C0
0,15,5,normali,C0_baseline,0.522,0.000
1,15,5,normali,C1_tags,0.772,0.250
2,15,5,normali,C3_dist_tags,0.800,0.278
3,15,5,normali,C4_dist_tags_joinconv,0.800,0.278
4,15,5,normali,C5_dist_tags_joinall,0.722,0.200
5,15,5,ambigue normali,C0_baseline,0.333,0.000
6,15,5,ambigue normali,C1_tags,0.867,0.533
7,15,5,ambigue normali,C3_dist_tags,0.900,0.567
8,15,5,ambigue normali,C4_dist_tags_joinconv,0.867,0.533
9,15,5,ambigue normali,C5_dist_tags_joinall,0.667,0.333


,top_k,top_j,subset,config,gold_recall,diff_vs_C0
0,15,5,normali,C0_baseline,0.522,0.000
1,15,5,normali,C1_tags,0.772,0.250
2,15,5,normali,C3_dist_tags,0.800,0.278
3,15,5,normali,C4_dist_tags_joinconv,0.800,0.278
4,15,5,normali,C5_dist_tags_joinall,0.722,0.200
5,15,5,ambigue normali,C0_baseline,0.333,0.000
6,15,5,ambigue normali,C1_tags,0.867,0.533
7,15,5,ambigue normali,C3_dist_tags,0.900,0.567
8,15,5,ambigue normali,C4_dist_tags_joinconv,0.867,0.533
9,15,5,ambigue normali,C5_dist_tags_joinall,0.667,0.333


quota di query con contesto CORTO (< top_j chunk: categoria 1 vuota, distanze non partite), regime (5,8), con distanze:


config
C0_baseline              0.00
C1_tags                  0.07
C3_dist_tags             0.07
C4_dist_tags_joinconv    0.07
C5_dist_tags_joinall     0.07
Name: retrieved_ids, dtype: float64